In [1]:
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

import pickle
from pathlib import Path

import sys
base_path = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(base_path / '2_Propensities'))
sys.path.insert(0, str(base_path / '4_Baselines' / '4.1_Matrix_Factorization'))

import SASRec_class as sasrec
import MF_class as MF


# 1 Load MF Model

In [ ]:
base_artifacts = Path.cwd().resolve().parents[1] / 'CausalI2I_artifacts'

In [3]:
n_users = 5000
n_items = 3000
n_tastes = 20

taste_dict = {
    t: np.arange(t, n_items, n_tastes)
    for t in range(n_tastes)
}

print(f"Maximal number of movies in a taste group: {max([len(taste_dict[g]) for g in range(n_tastes)]):,}")

Maximal number of movies in a taste group: 150


# 2 Generate Oracle

In [4]:
# Choose a random set of items to be the "causers" of the next item in the sequence.
rng = np.random.default_rng(42)
causers = rng.choice(
    n_items,
    size=int(0.5 * n_items),
    replace=False
)

# Choose a random set of items to be the "effects" of the "causers", ensuring that no item is its own effect.
effects = []
for cause in causers:
    taste_group = cause % n_tastes
    effect = rng.choice(taste_dict[taste_group])
    while effect == cause:
        effect = rng.choice(taste_dict[taste_group])
    effects.append(effect)

# Create a dictionary that maps each "causer" item to its 
# corresponding "next" item and the probability of that transition.
causal_dict = {
    cause: {
        "next": next_item, 
        "probability": rng.uniform(0.0, 0.3)} 
    for cause, next_item in zip(causers, effects)
}

print(f"Generated {len(causal_dict):,} causal pairs out of {n_items:,} items.")

# Create a DataFrame to store the causal pairs and their probabilities.
oracle_df = pd.DataFrame(
    {
        'cause_id': causers,
        'effect_id': [causal_dict[cause]['next'] for cause in causers],
        'probability': [causal_dict[cause]['probability'] for cause in causers]
    }
)

Generated 1,500 causal pairs out of 3,000 items.


# 3 Generate Data

In [5]:
def generate_movie_sequence(
    user_id,
    sequence_length,
):
    rng = np.random.default_rng(user_id)

    all_movies = np.arange(n_items)

    user_taste = rng.choice(n_tastes)
    movie_pool = taste_dict[user_taste]

    watched = np.zeros(n_items, dtype=bool)
    sequence = []

    for t in range(sequence_length):

        forced_movie = None

        if t > 0:
            previous_movie = sequence[-1]

            if previous_movie in causers:
                right = causal_dict[previous_movie]['next']

                if not watched[right]:
                    probability = causal_dict[previous_movie]['probability']

                    if rng.random() < probability:
                        forced_movie = right

        if forced_movie is not None:
            next_movie = forced_movie

        elif rng.random() < 0.1:
            available = all_movies[~watched[all_movies]]
            next_movie = rng.choice(available)

        else:
            available = movie_pool[~watched[movie_pool]]
            next_movie = rng.choice(available)

        sequence.append(next_movie)
        watched[next_movie] = True

    return sequence

In [6]:
sequences = {}
for u_id in tqdm(range(n_users), smoothing=0.05):
    sequences[u_id] = generate_movie_sequence(
        user_id=u_id,
        sequence_length=150,
    )

  0%|          | 0/5000 [00:00<?, ?it/s]

In [7]:
user_id = []
item_id = []
timestamp = []
k = len(sequences[0])

for u_id in range(n_users):
    user_id += [u_id] * k
    item_id += sequences[u_id]
    timestamp += list(range(k))

simulation_data = pd.DataFrame({
    'user_id': user_id,
    'item_id': item_id,
    'timestamp': timestamp,
})

print(f"Generated simulation data with shape: {len(simulation_data):,} rows.")

Generated simulation data with shape: 750,000 rows.


# 4 Choose 10K Pairs

In [8]:
# Make a pivot table
data_pairs = simulation_data.copy()
data_pairs['interaction'] = 1
pivot_table = data_pairs.pivot(index='user_id', columns='item_id', values='interaction').fillna(0)

# Make a correlation matrix
X = pivot_table.values
mean = X.mean(axis=0)
std  = np.maximum(X.std(axis=0, ddof=1), 1e-8)
M = (X - mean) / std
corr_mat = (M.T @ M) / (M.shape[0] - 1)
np.fill_diagonal(corr_mat, 0)

# Find the top 10,000 pairs of items with the highest correlation values.
correlated_pairs_flat_coordinates = corr_mat.flatten().argsort()[::-1][:10000]
n_corr_items = corr_mat.shape[1]
correlated_pairs_cols = [
    (i // n_corr_items, i % n_corr_items)
    for i in correlated_pairs_flat_coordinates
]
col2item = {i: col for i, col in enumerate(pivot_table.columns)}
correlated_pairs_ids = [
    (col2item[i], col2item[j])
    for i, j in correlated_pairs_cols
]

In [9]:
causal_pairs = [
    (cause, effect)
    for cause, effect in zip(causers, effects)
]

chosen_pairs_ids = []
for pair in causal_pairs:
    cause, effect = pair
    chosen_pairs_ids.extend([(cause, effect), (effect, cause)])
for pair in correlated_pairs_ids:
    if len(chosen_pairs_ids) >= 10000:
        break
    chosen_pairs_ids.append(pair)

# 5 Save Results

In [10]:
simulation_path = base_artifacts / 'Datasets' / 'Simulation'

simulation_data.to_csv(simulation_path / 'simulation_data.csv', index=False)

oracle_df.to_csv(simulation_path / 'oracle.csv', index=False)

rng = np.random.default_rng(42)
test_users = rng.choice(n_users, size=int(0.2 * n_users), replace=False)
with open(simulation_path / 'test_users.pkl', 'wb') as f:
    pickle.dump(test_users, f)

with open(simulation_path / 'chosen_pairs_ids.pkl', 'wb') as f:
    pickle.dump(chosen_pairs_ids, f)